# Cápsula 3 · Estudo de eventos: a empresa do seu grupo reagiu à notícia?

**ED0139 · Finanças Corporativas · 2026-2 · Prof. Sérgio Cardoso · UFC · Ciências Atuariais**

**Nome:** _(escreva aqui)_   **Matrícula:** _(escreva aqui)_   **Grupo e empresa:** _(escreva aqui)_

Entrega pelo SIGAA, tarefa "Cápsula 3", até **27/11/2026, 23h59**: este notebook **executado**, com as
respostas nas células marcadas. O enunciado completo está no PDF da Cápsula 3.

**Como usar.** Preencha a célula de parâmetros (seção 1), rode tudo (no Colab: *Ambiente de execução →
Executar tudo*), confira os dias "ex" (seção 3), rode de novo e responda às seis questões.

## 1. Parâmetros: preencha só esta célula

In [ ]:
# Código da ação da empresa do seu grupo, exatamente como abaixo:
# RAIL3 (Rumo), WEGE3 (WEG), VALE3 (Vale), ABEV3 (Ambev), BBSE3 (BB Seguridade),
# B3SA3 (B3), LREN3 (Lojas Renner), HAPV3 (Hapvida)
TICKER = ""

# Data em que a notícia foi divulgada, no formato "AAAA-MM-DD"
DATA_NOTICIA = ""

# A notícia saiu com o pregão fechado (à noite, de madrugada, fim de semana)? True ou False.
# Se True, o dia 0 é o pregão seguinte; se False, é o próprio dia.
PREGAO_FECHADO = True

# Uma linha sobre a notícia e o link do documento (RI ou CVM)
DESCRICAO = ""

# Dias "ex" (dividendo ou JCP) dentro do período de estimação, no formato "AAAA-MM-DD".
# Deixe vazio na primeira rodada; preencha depois de conferir a seção 3.
EXCLUIR = []

## 2. Dados

In [ ]:
from pathlib import Path
from urllib.error import URLError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

AZUL, LARANJA, CINZA = "#005386", "#E77813", "#5E6E7A"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})
PCT = FuncFormatter(lambda v, _: f"{100*v:.0f}%".replace(".", ","))
BASE_DADOS = "https://sergiocardoso.pro.br/ufc/financas-corporativas/2026-2/dados"
ARQ = "precos/fechamentos_diarios_2021-2026_congelado_2026-08-24.csv"


def num(x, casas=2):
    return f"{x:.{casas}f}".replace(".", ",")


def pct(x, casas=2):
    return f"{x:.{casas}%}".replace(".", ",")


def le():
    try:
        return pd.read_csv(f"{BASE_DADOS}/{ARQ}", parse_dates=["data"])
    except (URLError, OSError, Exception):
        base = Path.cwd().resolve()
        for p in [base, *base.parents][:8]:
            if (p / "dados" / ARQ).is_file():
                return pd.read_csv(p / "dados" / ARQ, parse_dates=["data"])
        raise FileNotFoundError("sem rede e sem a pasta 'dados/' da disciplina por perto")


precos = le().set_index("data")
ret = precos.pct_change().iloc[1:]
# Hapvida: grupamento de 15 ações em 1 a partir de 06/06/2025; o salto de preço não é retorno.
GRUP = pd.Timestamp("2025-06-06")
ret.loc[GRUP, "HAPV3"] = precos.loc[GRUP, "HAPV3"] / (precos.shift(1).loc[GRUP, "HAPV3"] * 15) - 1

NOMES = {"RAIL3": "Rumo", "WEGE3": "WEG", "VALE3": "Vale", "ABEV3": "Ambev", "BBSE3": "BB Seguridade",
         "B3SA3": "B3", "LREN3": "Lojas Renner", "HAPV3": "Hapvida"}
L, G, W = 250, 10, 5     # 250 pregões de estimação, 10 de intervalo, janela de -5 a +5

assert TICKER in NOMES, "Preencha TICKER na seção 1 com um dos oito códigos."
assert DATA_NOTICIA, "Preencha DATA_NOTICIA na seção 1."
assert DESCRICAO, "Descreva a notícia em DESCRICAO na seção 1."

d = pd.Timestamp(DATA_NOTICIA)
pos = ret.index.searchsorted(d, side="right" if PREGAO_FECHADO else "left")
assert pos < len(ret), "A data está depois do fim dos dados (20/08/2026)."
T0 = ret.index[pos]
i0 = ret.index.get_loc(T0)
assert i0 >= L + G, f"Dia 0 cedo demais: o primeiro dia 0 possível é {ret.index[L + G].date()}."
assert i0 + W < len(ret), f"Dia 0 tarde demais: o último dia 0 possível é {ret.index[len(ret) - W - 1].date()}."
print(f"{NOMES[TICKER]} ({TICKER}) | notícia em {d.date()} | dia 0 = {T0.date()}")
print(f"período de estimação: {ret.index[i0 - G - L].date()} a {ret.index[i0 - G - 1].date()}")

## 3. Dias "ex": confira antes de confiar no resultado

Os preços **não são ajustados por proventos**. No dia "ex" de um dividendo ou JCP grande, a ação cai
sem que o mercado caia, e isso entra no período de estimação como se fosse risco da ação. A tabela
mostra os maiores resíduos negativos do modelo de mercado no período de estimação. Confira no site de
RI da empresa se algum é dia "ex"; se for, ponha a data em `EXCLUIR` (seção 1) e rode tudo de novo.

In [ ]:
def estudo(tk, t0, excluir=(), modelo="mercado"):
    """Mesma função do notebook da aula 11: devolve a janela de -5 a +5 e os parâmetros."""
    s = ret[[tk, "IBOV"]]
    i = s.index.get_loc(pd.Timestamp(t0))
    est = s.iloc[i - G - L:i - G]
    est = est[~est.index.isin(pd.to_datetime(list(excluir)))]
    if modelo == "mercado":
        beta, alfa = np.polyfit(est["IBOV"], est[tk], 1)
        ddof = 2
    else:
        alfa, beta, ddof = 0.0, 1.0, 0
    resid = est[tk] - (alfa + beta * est["IBOV"])
    sd = resid.std(ddof=ddof)
    jan = s.iloc[i - W:i + W + 1].copy()
    jan["t"] = range(-W, W + 1)
    jan["esperado"] = alfa + beta * jan["IBOV"]
    jan["AR"] = jan[tk] - jan["esperado"]
    jan["CAR"] = jan["AR"].cumsum()
    return jan, dict(alfa=alfa, beta=beta, sd=sd, n=len(est), resid=resid,
                     ini=est.index[0].date(), fim=est.index[-1].date())


_, p_bruto = estudo(TICKER, T0)
suspeitos = p_bruto["resid"].nsmallest(8).to_frame("resíduo")
suspeitos["retorno da ação"] = ret.loc[suspeitos.index, TICKER]
suspeitos["retorno do Ibovespa"] = ret.loc[suspeitos.index, "IBOV"]
suspeitos.index = suspeitos.index.date
display(suspeitos.map(pct))
print("Já excluídos:", EXCLUIR if EXCLUIR else "nenhum")

## 4. O estudo de eventos

Modelo de mercado estimado no período de estimação, sem os dias excluídos:

$$R_{i,t} = \alpha_i + \beta_i R_{m,t} + \varepsilon_{i,t} \qquad
AR_{i,t} = R_{i,t} - (\hat\alpha_i + \hat\beta_i R_{m,t}) \qquad CAR = \sum AR$$

In [ ]:
jan, par = estudo(TICKER, T0, excluir=EXCLUIR)
print(f"estimação: {par['ini']} a {par['fim']}, {par['n']} pregões")
print("alfa =", pct(par["alfa"], 3), "| beta =", num(par["beta"], 3), "| desvio do resíduo =", pct(par["sd"]))
display(jan.set_index("t")[[TICKER, "IBOV", "esperado", "AR", "CAR"]].map(pct))

d0 = jan[jan.t == 0].iloc[0]
t_d0 = d0.AR / par["sd"]
antes, dia0 = jan.AR.iloc[:W].sum(), d0.AR
depois = jan.AR.iloc[W + 1:].sum()
print(f"\ndia 0: AR = {pct(dia0)} | estatística t = {num(t_d0, 1)}")
print(f"CAR antes (-5 a -1) = {pct(antes)} | dia 0 = {pct(dia0)} | depois (+1 a +5) = {pct(depois)} | total = {pct(jan.CAR.iloc[-1])}")
print(f"um desvio de 5 pregões, para comparar com 'antes' e 'depois': {pct(par['sd'] * np.sqrt(5))}")

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.bar(jan.t, jan.AR, color=AZUL, alpha=.6, label="anormal do dia")
ax.plot(jan.t, jan.CAR, color=LARANJA, marker="o", lw=2.2, label="acumulado")
ax.axhline(0, color=CINZA, lw=.8); ax.yaxis.set_major_formatter(PCT)
ax.set_xlabel(f"pregões em relação a {T0.date()}"); ax.legend(loc="best")
ax.set_title(f"{NOMES[TICKER]}: {DESCRICAO[:70]}", fontsize=10)
plt.tight_layout(); plt.show()

## 5. Robustez: as escolhas que mudam o número

In [ ]:
linhas = []
for modelo in ("mercado", "ajustado"):
    for rotulo, exc in (("excluídos", EXCLUIR), ("mantidos", [])):
        j, p = estudo(TICKER, T0, excluir=exc, modelo=modelo)
        linhas.append({"modelo": modelo, "dias ex": rotulo, "beta": round(p["beta"], 3),
                       "AR do dia 0": pct(j.AR.iloc[W]), "CAR de -5 a +5": pct(j.CAR.iloc[-1])})
display(pd.DataFrame(linhas).drop_duplicates())

## 6. Respostas

Responda nas células abaixo, com os números das seções 3 a 5.

**1. O evento.** Qual foi a notícia, quando e a que horas foi divulgada (com o link do documento), e por que o dia 0 é o que você escolheu.

_Sua resposta:_ 

**2. O retorno normal.** Período de estimação, alfa, beta e desvio do resíduo; em uma frase, o que o beta diz sobre a ação.

_Sua resposta:_ 

**3. Os dias "ex".** Quais dias você excluiu e por quê. Se não excluiu nenhum, mostre que conferiu.

_Sua resposta:_ 

**4. O dia 0.** Retorno anormal e estatística t do dia 0. A reação se distingue do ruído diário? O sinal faz sentido com a notícia?

_Sua resposta:_ 

**5. Antes e depois.** CAR antes, no dia 0 e depois, à luz da forma semiforte: houve sinal de vazamento? Reação lenta ou exagerada?

_Sua resposta:_ 

**6. Robustez e conclusão.** O que muda com o modelo ajustado e com os dias "ex" mantidos, o que não muda, e uma frase de conclusão.

_Sua resposta:_ 